In [0]:
 # Databricks notebook source

# ============================================================
# 02_SILVER
# Pipeline de Dados - Inside Airbnb São Paulo
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
 # COMMAND ----------

# ============================================================
# LEITURA DA BRONZE
# ============================================================

BRONZE_TABLE = "workspace.default.bronze_listings"
SILVER_TABLE = "workspace.default.silver_listings"

df_bronze = spark.table(BRONZE_TABLE)

print("Registros Bronze:", df_bronze.count())
print("Atributos Bronze:", len(df_bronze.columns))

In [0]:
 # COMMAND ----------

# ============================================================
# PADRONIZAÇÃO E TIPAGEM
# ============================================================

df_silver = df_bronze.select(
    F.col("id").cast("long").alias("id"),

    F.col("neighbourhood_cleansed")
        .cast("string")
        .alias("neighbourhood_cleansed"),

    F.col("room_type")
        .cast("string")
        .alias("room_type"),

    F.regexp_replace(
        F.regexp_replace(
            F.col("price"),
            r"[$]",
            ""
        ),
        ",",
        ""
    )
    .cast("double")
    .alias("price"),

    F.col("accommodates")
        .cast("int")
        .alias("accommodates"),

    F.col("bedrooms")
        .cast("double")
        .alias("bedrooms"),

    F.col("beds")
        .cast("double")
        .alias("beds"),

    F.col("availability_30")
        .cast("string")
        .alias("availability_30"),

    F.col("availability_60")
        .cast("string")
        .alias("availability_60"),

    F.col("availability_90")
        .cast("string")
        .alias("availability_90"),

    F.col("availability_365")
        .cast("int")
        .alias("availability_365"),

    F.col("number_of_reviews")
        .cast("int")
        .alias("number_of_reviews"),

    F.col("review_scores_rating")
        .cast("double")
        .alias("review_scores_rating"),

    F.col("host_is_superhost")
        .cast("string")
        .alias("host_is_superhost"),

    F.col("instant_bookable")
        .cast("string")
        .alias("instant_bookable"),

    F.col("last_scraped")
        .cast("string")
        .alias("last_scraped")
)

print("Registros Silver:", df_silver.count())
print("Atributos Silver:", len(df_silver.columns))

display(df_silver.limit(10))

In [0]:
 # COMMAND ----------

# ============================================================
# FLAGS DE QUALIDADE
# ============================================================

df_silver_quality = (
    df_silver

    .withColumn(
        "flag_price_null",
        F.when(
            F.col("price").isNull(),
            1
        ).otherwise(0)
    )

    .withColumn(
        "flag_price_invalid",
        F.when(
            F.col("price").isNotNull()
            & (F.col("price") <= 0),
            1
        ).otherwise(0)
    )

    .withColumn(
        "flag_bedrooms_null",
        F.when(
            F.col("bedrooms").isNull(),
            1
        ).otherwise(0)
    )

    .withColumn(
        "flag_beds_null",
        F.when(
            F.col("beds").isNull(),
            1
        ).otherwise(0)
    )

    .withColumn(
        "flag_rating_null",
        F.when(
            F.col("review_scores_rating").isNull(),
            1
        ).otherwise(0)
    )
)

display(df_silver_quality.limit(10))

In [0]:
 # COMMAND ----------

# ============================================================
# PERSISTÊNCIA SILVER
# ============================================================

(
    df_silver_quality.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(SILVER_TABLE)
) 

print(
    f"Tabela Silver criada/atualizada: {SILVER_TABLE}"
)

In [0]:
 # COMMAND ----------

# ============================================================
# AUDITORIA FINAL DA SILVER
# ============================================================

df_silver_final = spark.table(SILVER_TABLE)

print(
    "Registros Silver:",
    df_silver_final.count()
)

print(
    "Atributos Silver:",
    len(df_silver_final.columns)
)

display(
    df_silver_final.select(
        F.count("*").alias("total"),
        F.sum("flag_price_null").alias("price_null"),
        F.sum("flag_price_invalid").alias("price_invalid"),
        F.sum("flag_bedrooms_null").alias("bedrooms_null"),
        F.sum("flag_beds_null").alias("beds_null"),
        F.sum("flag_rating_null").alias("rating_null")
    )
)

In [0]:
 # COMMAND ----------

# ============================================================
# DUPLICIDADE
# ============================================================

total = df_silver_final.count()

distintos = (
    df_silver_final
    .select("id")
    .distinct()
    .count()
)

print("Total:", total)
print("IDs distintos:", distintos)
print("Duplicados:", total - distintos)